# Cross-dataset example: flights + OHLCV

This notebook demonstrates the full alt-data research workflow:
1. Load daily flight features for an airport (`KJFK`).
2. Load OHLCV for the airport's associated airline (`DAL`).
3. Join the two on trading day via `DataHub`.
4. Plot a simple comparison of net flow vs returns.

In [ ]:
import pandas as pd

from alt_data.query.datahub import DataHub
from alt_data.query.mappings import AssetMapper

pd.set_option('display.max_columns', 20)

hub = DataHub()
mapper = AssetMapper()

## 1. Inspect the mapping layer

In [ ]:
print('Airports mapped:', mapper.all_airports())
print('KJFK -> tickers:', mapper.tickers_for_airport('KJFK'))
print('DAL serves airports:', mapper.airports_for_ticker('DAL'))

## 2. Flight data (daily aggregated)

In [ ]:
flights = hub.get_flight_data(
    airport='KJFK',
    start='2024-01-01',
    end='2024-02-01',
)
flights.head()

## 3. Stored features (long format pivoted to wide)

In [ ]:
features = hub.get_flight_data(
    airport='KJFK',
    start='2024-01-01',
    end='2024-02-01',
    features=['net_flow', 'arrivals', 'departures', 'hub_connectivity'],
)
features.head()

## 4. OHLCV for the mapped airline

In [ ]:
ohlcv = hub.get_ohlcv_data(
    ticker='DAL',
    start='2024-01-01',
    end='2024-02-01',
)
ohlcv.head()

## 5. Cross-dataset join

`get_joined_data` pulls both sides and aligns on the OHLCV trading-day index.
Shift the flight frame by `shift_days=1` to study whether today's flight
volume predicts tomorrow's DAL return.

In [ ]:
joined = hub.get_joined_data(
    airport='KJFK',
    ticker='DAL',
    start='2024-01-01',
    end='2024-02-01',
    features=['net_flow', 'arrivals', 'departures'],
    shift_days=1,
)
joined['close_ret'] = joined['close'].pct_change()
joined[['net_flow', 'close', 'close_ret']].head(10)

## 6. Simple signal: correlation of net flow and next-day return

In [ ]:
corr = joined[['net_flow', 'close_ret']].dropna().corr().iloc[0, 1]
print(f'corr(net_flow(t), return(t+1)) = {corr:.4f}')

## 7. Multi-ticker basket around the airport

In [ ]:
basket = hub.get_airport_basket(
    airport='KJFK',
    start='2024-01-01',
    end='2024-02-01',
    feature='net_flow',
)
basket.head()